TASK 2 NOTEBOOK

Downloading Libraries and Imports

In [1]:

!pip install -q --upgrade "pillow<11.0.0" transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import numpy as np
from functools import partial
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import notebook_login
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from PIL import Image
from tqdm import tqdm
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    Trainer,
    TrainingArguments,
)


Setting up environment

In [2]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "main",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    # If the repository folder already exists, reference it safely
    if repo_root.exists():
        print(f"Repository directory already exists at: {repo_root}")
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent


print(f"Setup Complete. REPO_ROOT: {repo_root}")

# Check GPU and device availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Configure system paths for absolute module imports
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

# 4. Import custom project modules cleanly
from data.generation import (
    build_sample,
    generate_dataset,
)

from data.utilities import (
    load_lichess_csv,
    upload_dataset_to_hub,
    authenticate_hf,
)

from eval.utilities import (
   calculate_fen_exact_match,
   calculate_levenshtein_metrics,
   calculate_square_by_square_accuracy,
   evaluate_chessboard_model_task_2,
   preprocess_function,
   get_patch_reordering_indices,
   reorder_chessboard_image,
   finetune_and_push_chessboard_model,
)

print("All custom modules and eval utilities imported successfully!")

Cloning repository from https://github.com/Aivon99/BigDataAndTextMiningProject.git...
Setup Complete. REPO_ROOT: /content/BigDataAndTextMiningProject
Using device: cuda
GPU: NVIDIA A100-SXM4-40GB
All custom modules and eval utilities imported successfully!


Dowloading dataset for task 2 from HuggingFace repo

In [3]:
print("Verifying Authentication to Hugging Face...")
notebook_login()

dataset_name = "bdatm-project/dataset_task2"
print(f"Downloading dataset '{dataset_name}'...")

dataset_task2 = load_dataset(dataset_name)

print("\nDataset loaded successfully!")
print(dataset_task2)
print("\nStructure sample of train split:")
print(dataset_task2["train"][0])

Verifying Authentication to Hugging Face...


Resolving data files:   0%|          | 0/33 [00:00<?, ?it/s]

train/sample_000002/board.png: reconstructing file:   0%|          |  0.00B / 42.2kB            

train/sample_000003/board.png: reconstructing file:   0%|          |  0.00B / 24.5kB            

train/sample_000011/board.png: reconstructing file:   0%|          |  0.00B / 38.9kB            

...

metadata.jsonl:   0%|          | 0.00/2.14k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/32 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4 [00:00<?, ? examples/s]


Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 32
    })
    validation: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 4
    })
    test: Dataset({
        features: ['sample_id', 'puzzle_id', 'task', 'fen', 'prompt', 'target', 'image', 'file_name_t1'],
        num_rows: 4
    })
})

Structure sample of train split:
{'sample_id': 'sample_000000', 'puzzle_id': '2GDeK', 'task': 'task2', 'fen': 'r1b2rk1/ppRq3p/3p2p1/3PPp2/8/3B1NP1/2Q2K1P/8 b - - 1 30', 'prompt': 'You are a specialized model for chessboard understanding.\nYour goal is to predict the last move played on the board based on the highlighted squares.\nInput:\n- Board Image: The visual representation with the last move highlighted.\nOutput Format:\nReturn only the move in Standard Algebraic Notation (SA

## Vanilla Model

Loading model

In [4]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("model and Processor loaded correctly!")

Loading model Qwen/Qwen3.5-0.8B...


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/50.9k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 1.75GB            

...

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model and Processor loaded correctly!


Test with baseline on a single sample

In [5]:
# Grab the first test sample for Task 2
test_sample = dataset_task2["test"][0]

ground_truth_move = test_sample["target"]
task_prompt = test_sample["prompt"]
sample_id = test_sample["sample_id"]
board_image = test_sample["image"]

print(f"Sample ID: {sample_id}")
print(f"Prompt provided to the model:\n{task_prompt}\n")
print(f"Real Move (Ground Truth SAN): {ground_truth_move}\n")

# Prepare the multimodal input format for the model
chat_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": board_image},
            {"type": "text", "text": task_prompt},
        ]
    }
]

# Apply the processor's chat template
formatted_text = processor.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

# Tokenize inputs and move them to the GPU device
model_inputs = processor(
    text=[formatted_text],
    images=board_image,
    padding=True,
    return_tensors="pt"
).to(model_vanilla.device)

# Generate the zero-shot prediction
print("Generating zero-shot prediction for Task 2...")
with torch.no_grad():
    # SAN notation is short, so we can use a smaller max_new_tokens value
    output_token_ids = model_vanilla.generate(**model_inputs, max_new_tokens=16)

# Trim prompt tokens from the generated output
trimmed_output_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, output_token_ids)
]
predicted_move = processor.batch_decode(
    trimmed_output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
)[0]

print(f"Predicted Move (Zero-Shot): {predicted_move.strip()}")

Sample ID: sample_000000
Prompt provided to the model:
You are a specialized model for chessboard understanding.
Your goal is to predict the last move played on the board based on the highlighted squares.
Input:
- Board Image: The visual representation with the last move highlighted.
Output Format:
Return only the move in Standard Algebraic Notation (SAN).

Real Move (Ground Truth SAN): Kg5

Generating zero-shot prediction for Task 2...
Predicted Move (Zero-Shot): Let's analyze the board.

The highlighted square is **g6** (


Testing vanilla model on the whole dataset using the function *evaluate_chessboard_model_task_1*

In [6]:
# Call the evaluation function for the vanilla model
vanilla_results_df, vanilla_summary_df = evaluate_chessboard_model_task_2(
    model=model_vanilla,
    processor=processor,
    dataset_split=dataset_task2["test"],
    model_name="Vanilla Qwen2.5-VL (Zero-Shot)"
)

# Initialize the global comparison DataFrame with the vanilla results
all_models_results = vanilla_summary_df

print("\nExample of results obtained form the evaluation:")
display(vanilla_results_df.head())

print("\nComparative Summary DataFrame (all_models_results):")
display(all_models_results)

Evaluating Vanilla Qwen2.5-VL (Zero-Shot) (Task 2): 100%|██████████| 4/4 [00:04<00:00,  1.02s/it]


Evaluation completed for Vanilla Qwen2.5-VL (Zero-Shot)! Results saved to task2_vanilla_qwen2.5-vl_(zero-shot)_results.csv.

Example of results obtained form the evaluation:


,sample_id,ground_truth,predicted,exact_match
0,sample_000000,Kg5,Let's analyze the board.\n\nThe highlighted sq...,0
1,sample_000001,Nxd4,The knight on c1 is in check. The only legal m...,0
2,sample_000002,Rxb2,"The piece at b2 is a white pawn, and the highl...",0
3,sample_000003,Nxe6,The knight at c1 is attacked by the pawn at f2...,0



Comparative Summary DataFrame (all_models_results):


,model_name,exact_match
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0


## Vanilla + LoRA

Preprocessing dataset and finetuning

In [7]:
# 1. Preprocessing dataset
print("Applying preprocessing to datasets...")
tokenized_train = dataset_task2["train"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task2["train"].column_names,
)
tokenized_val = dataset_task2["validation"].map(
    partial(preprocess_function, processor=processor),
    remove_columns=dataset_task2["validation"].column_names,
)

# 2. Configure PEFT and LoRA parameters
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
)

# Apply LoRA to model vanilla and saving the new one into lora_model
lora_model = get_peft_model(model_vanilla, peft_config)
lora_model.print_trainable_parameters()

# 3. Define Training Arguments
training_args = TrainingArguments(
    output_dir="./qwen_task2_lora_output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=2,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
)

# 4. Initialize the Trainer using 'lora_model'
trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
)

# 5. Start Fine-Tuning
print("Starting LoRA Supervised Fine-Tuning for Task 2...")
trainer.train()

Applying preprocessing to datasets...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

trainable params: 6,389,760 || all params: 859,375,680 || trainable%: 0.7435
Starting LoRA Supervised Fine-Tuning for Task 2...


Epoch,Training Loss,Validation Loss
1,No log,17.488049
2,No log,16.672985


TrainOutput(global_step=8, training_loss=17.497068405151367, metrics={'train_runtime': 73.7696, 'train_samples_per_second': 0.868, 'train_steps_per_second': 0.108, 'total_flos': 118618822017024.0, 'train_loss': 17.497068405151367, 'epoch': 2.0})

Saving model on hugging face

In [8]:
# 6. Save weights to Hugging Face folder
hf_org_prefix = "bdatm-project"
repo_id_standard = f"{hf_org_prefix}/qwen-task2-standard-lora"

print(f"Pushing standard LoRA model and processor to Hugging Face Hub: {repo_id_standard}...")

trainer.model.push_to_hub(
    repo_id_standard,
    commit_message="Training complete for standard LoRA baseline on Task 2 (Move Prediction)"
)
processor.push_to_hub(
    repo_id_standard
)

print("Fine-tuning for Task 2 completed and weights successfully uploaded to Hugging Face Hub!")



Pushing standard LoRA model and processor to Hugging Face Hub: bdatm-project/qwen-task2-standard-lora...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 18.0kB / 25.6MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpfq4qeje1/tokenizer.json: 100%|##########| 20.0MB / 20.0MB            

Fine-tuning for Task 2 completed and weights successfully uploaded to Hugging Face Hub!


Loading model and evaluation

In [9]:
# 7. Loading Model from hugging face and evaluate model using predefined functions
print("\nLoading standard LoRA model from Hugging Face for evaluation...")

# Load fresh base model instance
base_eval_model = AutoModelForImageTextToText.from_pretrained(
    "Qwen/Qwen3.5-0.8B",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# Load the LoRA weights directly from the Hub repository
standard_lora_eval_model = PeftModel.from_pretrained(base_eval_model, repo_id_standard)

print("Evaluating the Hugging Face LoRA model on the test set for Task 2...")
lora_results_df, lora_summary_df = evaluate_chessboard_model_task_2(
    model=standard_lora_eval_model,
    processor=processor,
    dataset_split=dataset_task2["test"], # <-- Corretto da task1 a task2
    model_name="Qwen + LoRA Fine-Tuning (from HF)"
)

# Append the new metrics to the global comparison DataFrame
all_models_results = pd.concat([all_models_results, lora_summary_df], ignore_index=True)

print("\nUpdated Comparative Summary Table (all_models_results):")
display(all_models_results)


Loading standard LoRA model from Hugging Face for evaluation...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Evaluating the Hugging Face LoRA model on the test set for Task 2...


Evaluating Qwen + LoRA Fine-Tuning (from HF) (Task 2): 100%|██████████| 4/4 [00:05<00:00,  1.30s/it]


Evaluation completed for Qwen + LoRA Fine-Tuning (from HF)! Results saved to task2_qwen_+_lora_fine-tuning_(from_hf)_results.csv.

Updated Comparative Summary Table (all_models_results):


,model_name,exact_match
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0
1,Qwen + LoRA Fine-Tuning (from HF),0.0


## Reordering patches - Advanced models

Checking function *get_patch_reordering_indices()*

In [10]:
for strat in ["raster", "zigzag", "spiral", "file_wise"]:
    order_map = get_patch_reordering_indices(strategy=strat)
    print(f"Strategy '{strat}' first 10 patch indices: {order_map[:10]}")

Strategy 'raster' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Strategy 'zigzag' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 15, 14]
Strategy 'spiral' first 10 patch indices: [0, 1, 2, 3, 4, 5, 6, 7, 15, 23]
Strategy 'file_wise' first 10 patch indices: [0, 8, 16, 24, 32, 40, 48, 56, 1, 9]


Let's evalaute the model we created before using three different patch ordering: zigzag, spiral and file-wise. We'll use the function ***reorder_chessboard_image*** defined in *src/eval/utilities.py*

In [11]:
# ==========================================
# Training-Free Benchmark Loop for REOrder (Task 2)
# ==========================================

# Define the strategies you want to benchmark (as outlined in the project specs)
strategies_to_test = ["zigzag", "spiral", "file_wise"]

# Choose the model to test (we use the fine-tuned LoRA model)
model_to_evaluate = lora_model

for strat in strategies_to_test:
    print(f"\nEvaluating strategy (Training-Free): {strat.upper()}...")

    # 1. Apply the reordering function to the images in the test set for Task 2
    reordered_test_split = dataset_task2["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 2. Run the evaluation function on the reordered test split using Task 2 evaluator
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_2(
        model=model_to_evaluate,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - TF)"
    )

    # 3. Append the results to your global comparison table
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table for Task 2 (Including Training-Free Strategies) ---")
display(all_models_results)


Evaluating strategy (Training-Free): ZIGZAG...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (Zigzag - TF) (Task 2): 100%|██████████| 4/4 [00:05<00:00,  1.30s/it]


Evaluation completed for Qwen + LoRA (Zigzag - TF)! Results saved to task2_qwen_+_lora_(zigzag_-_tf)_results.csv.

Evaluating strategy (Training-Free): SPIRAL...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

...


Evaluation completed for Qwen + LoRA (Spiral - TF)! Results saved to task2_qwen_+_lora_(spiral_-_tf)_results.csv.

Evaluating strategy (Training-Free): FILE_WISE...


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Evaluating Qwen + LoRA (File_wise - TF) (Task 2): 100%|██████████| 4/4 [00:04<00:00,  1.10s/it]


Evaluation completed for Qwen + LoRA (File_wise - TF)! Results saved to task2_qwen_+_lora_(file_wise_-_tf)_results.csv.

--- Final Comparative Summary Table for Task 2 (Including Training-Free Strategies) ---


,model_name,exact_match
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0
1,Qwen + LoRA Fine-Tuning (from HF),0.0
2,Qwen + LoRA (Zigzag - TF),0.0
3,Qwen + LoRA (Spiral - TF),0.0
4,Qwen + LoRA (File_wise - TF),0.0


As highlighted in the summary table, applying unconventional patch reordering strategies (such as Zigzag, Spiral, or File-wise) in a "Training-Free" (TF) manner leads to a performance drop, resulting in an increased Character Error Rate (CER) and Levenshtein distance compared to the standard raster-scan baseline.

To truly reap the benefits of the REOrder methodology, we must proceed with Supervised Fine-Tuning (SFT) directly on the pre-reordered dataset. This will allow the model to adapt its weights and attention layers to the new spatial serialization strategy.

Finetuning the new models on the dataset using function **finetune_and_push_chessboard_model()** defined in *src/eval/utilities.py*. This function directly upload the models on hugging face

In [12]:
strategies_to_train = ["zigzag", "spiral", "file_wise"]

# Dictionary to store the trained models in memory
trained_reordered_models = {}

for strat in strategies_to_train:
    print(f"\nTraining model with strategy: {strat.upper()} for Task 2...")
    trained_reordered_models[strat] = finetune_and_push_chessboard_model(
        strategy_name=strat,
        dataset=dataset_task2,
        processor=processor,
        peft_config=peft_config,
        task="task2",
    )

print("\nAll reordered models for Task 2 have been successfully trained and pushed to Hugging Face!")


Training model with strategy: ZIGZAG for Task 2...

Starting pipeline for TASK: TASK2 | STRATEGY: ZIGZAG
Applying zigzag reordering to datasets for task2...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Preprocessing datasets...


Map:   0%|          | 0/32 [00:00<?, ? examples/s]

...

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpm_3_u34v/tokenizer.json:  80%|#######9  | 16.0MB / 20.0MB            

Finished! Successfully uploaded to Hub: bdatm-project/qwen-task2-file_wise-lora

All reordered models for Task 2 have been successfully trained and pushed to Hugging Face!


Evaluating models using function **evaluate_chessboard_model_task_1()** defined in *src/eval/utilities.py*.

Models are downloaded from the hugging face repo.

In [13]:
strategies_to_evaluate = ["zigzag", "spiral", "file_wise"]
hf_org_prefix = "bdatm-project"

for strat in strategies_to_evaluate:
    print(f"\nLoading and evaluating SFT model for strategy: {strat.upper()} from Hugging Face (Task 2)...")

    # 1. Load fresh base model instance
    base_eval_model = AutoModelForImageTextToText.from_pretrained(
        "Qwen/Qwen3.5-0.8B",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto"
    )

    # 2. Load the specific LoRA weights from Hugging Face Hub for Task 2
    repo_id_source = f"{hf_org_prefix}/qwen-task2-{strat}-lora"
    model_to_eval = PeftModel.from_pretrained(base_eval_model, repo_id_source)

    # 3. Apply the specific patch reordering to the test split images for Task 2
    reordered_test_split = dataset_task2["test"].map(
        lambda sample: {
            "image": reorder_chessboard_image(sample["image"], strategy=strat, grid_size=8)
        }
    )

    # 4. Run the evaluation utility function for Task 2
    strat_results_df, strat_summary_df = evaluate_chessboard_model_task_2(
        model=model_to_eval,
        processor=processor,
        dataset_split=reordered_test_split,
        model_name=f"Qwen + LoRA ({strat.capitalize()} - SFT)"
    )

    # 5. Append results to the global comparison DataFrame
    all_models_results = pd.concat([all_models_results, strat_summary_df], ignore_index=True)

print("\n--- Final Comparative Summary Table for Task 2 (Loaded from Hub & Evaluated) ---")
display(all_models_results)


Loading and evaluating SFT model for strategy: ZIGZAG from Hugging Face (Task 2)...


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

...

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 25.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Evaluating Qwen + LoRA (File_wise - SFT) (Task 2): 100%|██████████| 4/4 [00:04<00:00,  1.13s/it]


Evaluation completed for Qwen + LoRA (File_wise - SFT)! Results saved to task2_qwen_+_lora_(file_wise_-_sft)_results.csv.

--- Final Comparative Summary Table for Task 2 (Loaded from Hub & Evaluated) ---


,model_name,exact_match
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0
1,Qwen + LoRA Fine-Tuning (from HF),0.0
2,Qwen + LoRA (Zigzag - TF),0.0
3,Qwen + LoRA (Spiral - TF),0.0
4,Qwen + LoRA (File_wise - TF),0.0
5,Qwen + LoRA (Zigzag - SFT),0.0
6,Qwen + LoRA (Spiral - SFT),0.0
7,Qwen + LoRA (File_wise - SFT),0.0


Displaying oredered results

In [14]:
sorted_results_df = all_models_results.sort_values(by="exact_match", ascending=False).reset_index(drop=True)

print("\n--- Comparative Summary Table for Task 2 (Ordered from Best to Worst by Exact Match) ---")
display(sorted_results_df)


--- Comparative Summary Table for Task 2 (Ordered from Best to Worst by Exact Match) ---


,model_name,exact_match
0,Vanilla Qwen2.5-VL (Zero-Shot),0.0
1,Qwen + LoRA Fine-Tuning (from HF),0.0
2,Qwen + LoRA (Zigzag - TF),0.0
3,Qwen + LoRA (Spiral - TF),0.0
4,Qwen + LoRA (File_wise - TF),0.0
5,Qwen + LoRA (Zigzag - SFT),0.0
6,Qwen + LoRA (Spiral - SFT),0.0
7,Qwen + LoRA (File_wise - SFT),0.0
